# 处理数据
- 将GPT评估结果附加到.tsv文件中

In [10]:
import json
import pandas as pd
import os

In [11]:
df = pd.read_csv("uncertainty_scores.tsv", sep='\t', dtype={'id': str})

with open("intermediate_data.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)


In [12]:
id_to_score_map = {}
for item in json_data:
    item_id = str(item.get('id'))
    score = item['output']['metric_score']['gpt']
    id_to_score_map[item_id] = score

In [13]:
new_col_name = 'gpt_score'
df[new_col_name] = df['id'].map(id_to_score_map)
missing_count = df[new_col_name].isna().sum()
if missing_count > 0:
    print(f"注意: 有 {missing_count} 行数据在 TSV 中存在但在 JSON 中未找到对应的 ID (填充为空)。")

In [14]:
df.to_csv("uncertainty_scores_with_gpt.tsv", sep='\t', index=False)

# 计算Pearson

In [15]:
import numpy as np
from sklearn.metrics import roc_auc_score

In [16]:
df = pd.read_csv('uncertainty_scores_with_gpt.tsv', sep='\t')
df = df[df['uncertainty_score'] != 0.0]
len(df)

570

In [17]:
# 确保数据是数值类型
df['uncertainty_score'] = pd.to_numeric(df['uncertainty_score'], errors='coerce')
df['gpt_score'] = pd.to_numeric(df['gpt_score'], errors='coerce')
df = df.dropna() # 再次移除转换数字失败的行

In [18]:
pearson_corr = df['uncertainty_score'].corr(df['gpt_score'], method='pearson')
pearson_corr

0.018093451683987155

# 计算AUROC

In [23]:
y_true = df['gpt_score'].values
y_score = df['uncertainty_score']
auroc = roc_auc_score(y_true, y_score)
auroc

0.49860435971441586